In [13]:
import sqlite3
import pandas as pd
import numpy as np
conn = sqlite3.connect('../data/sql/control_presupuestario.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,departamentos
1,categorias
2,periodos
3,presupuesto
4,ejecucion_real_raw
5,ingresos


In [2]:
df_raw = pd.read_sql('SELECT * FROM ejecucion_real_raw', conn)
df_raw.head()

,id_ejecucion,departamento,categoria,anio,mes,monto_ejecutado,fecha_registro
0,1,Atencion al Cliente,Nomina y Personal,2025,1,42692.08,2025-01-09
1,2,At. Cliente,Nomina y Personal,2025,2,43456.48,2025-02-21
2,3,Atencion al Cliente,Nomina y Personal,2025,3,"42,426.84 EUR",2025-03-15
3,4,ATENCION AL CLIENTE,Nomina y Personal,2025,4,43062.15,2025-04-05
4,5,Atencion al Cliente,Nomina y Personal,2025,5,43880.34,2025-05-06


In [3]:
mapa_departamentos = {
    'atencion al cliente': 'Atencion al Cliente', 'atencion cliente': 'Atencion al Cliente',
    'at. cliente': 'Atencion al Cliente',
    'ventas b2b': 'Ventas B2B', 'ventas corporativas b2b': 'Ventas B2B',
    'operaciones y logistica': 'Operaciones y Logistica', 'operaciones/logistica': 'Operaciones y Logistica',
    'marketing': 'Marketing', 'mercadeo': 'Marketing',
    'tecnologia y ti': 'Tecnologia y TI', 'ti': 'Tecnologia y TI', 'tecnologia/ti': 'Tecnologia y TI',
    'administracion y finanzas': 'Administracion y Finanzas', 'admin y finanzas': 'Administracion y Finanzas',
}

def normalizar_departamento(nombre):
    return mapa_departamentos.get(str(nombre).strip().lower(), str(nombre).strip())

df_raw['departamento_normalizado'] = df_raw['departamento'].apply(normalizar_departamento)

# Validacion: no deberia quedar ningun nombre sin mapear a la dimension
df_departamentos = pd.read_sql('SELECT * FROM departamentos', conn)
no_mapeados = set(df_raw['departamento_normalizado']) - set(df_departamentos['nombre'])
assert not no_mapeados, f'Quedaron sin mapear: {no_mapeados}'
print('Todos los departamentos se mapearon correctamente.')

Todos los departamentos se mapearon correctamente.


In [4]:
def limpiar_monto(valor):
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)
    texto = str(valor).strip().replace('EUR', '').replace('€', '').strip()
    if texto == '' or texto.upper() == 'N/D':
        return np.nan
    if ',' in texto and '.' in texto:
        texto = texto.replace(',', '')      # separador de miles: 4,894.70
    elif ',' in texto and '.' not in texto:
        texto = texto.replace(',', '.')     # formato europeo: 3880,15
    try:
        return float(texto)
    except ValueError:
        return np.nan

df_raw['monto_numerico'] = df_raw['monto_ejecutado'].apply(limpiar_monto)
print('Valores no reconstruibles (nulos tras conversion):', df_raw['monto_numerico'].isna().sum())

Valores no reconstruibles (nulos tras conversion): 10


In [6]:
df_raw['flag_valor_nulo'] = df_raw['monto_numerico'].isna()
df_raw['flag_valor_negativo'] = df_raw['monto_numerico'] < 0
df_raw['monto_corregido'] = df_raw['monto_numerico'].abs()

cols_dup = ['departamento_normalizado', 'categoria', 'anio', 'mes', 'monto_numerico', 'fecha_registro']
df_raw['flag_duplicado'] = df_raw.duplicated(subset=cols_dup, keep='first')

total = len(df_raw)
print('--- Resumen de calidad de datos ---')
print(f"Total filas raw: {total}")
print(f"Valores nulos/no numericos: {df_raw['flag_valor_nulo'].sum()} ({df_raw['flag_valor_nulo'].sum()/total*100:.1f}%)")
print(f"Valores negativos corregidos: {df_raw['flag_valor_negativo'].sum()} ({df_raw['flag_valor_negativo'].sum()/total*100:.1f}%)")
print(f"Filas duplicadas eliminadas: {df_raw['flag_duplicado'].sum()} ({df_raw['flag_duplicado'].sum()/total*100:.1f}%)")

--- Resumen de calidad de datos ---
Total filas raw: 241
Valores nulos/no numericos: 10 (4.1%)
Valores negativos corregidos: 5 (2.1%)
Filas duplicadas eliminadas: 1 (0.4%)


In [7]:
df_categorias = pd.read_sql('SELECT * FROM categorias', conn)
df_periodos = pd.read_sql('SELECT * FROM periodos', conn)

df_limpio = df_raw[~df_raw['flag_valor_nulo'] & ~df_raw['flag_duplicado']].copy()

df_limpio = df_limpio.merge(df_departamentos, left_on='departamento_normalizado', right_on='nombre')
df_limpio = df_limpio.merge(df_categorias, left_on='categoria', right_on='nombre', suffixes=('', '_cat'))
df_limpio = df_limpio.merge(df_periodos, on=['anio', 'mes'])

ejecucion_real_limpia = df_limpio[[
    'departamento_id', 'categoria_id', 'periodo_id', 'monto_corregido', 'flag_valor_negativo'
]].rename(columns={'monto_corregido': 'monto_ejecutado', 'flag_valor_negativo': 'fue_corregido_signo'})
ejecucion_real_limpia.insert(0, 'id_ejecucion', range(1, len(ejecucion_real_limpia) + 1))

print(f"Filas en dataset limpio: {len(ejecucion_real_limpia)} de {total} originales")
ejecucion_real_limpia.head()

Filas en dataset limpio: 230 de 241 originales


,id_ejecucion,departamento_id,categoria_id,periodo_id,monto_ejecutado,fue_corregido_signo
0,1,1,1,1,42692.08,False
1,2,1,1,2,43456.48,False
2,3,1,1,3,42426.84,False
3,4,1,1,4,43062.15,False
4,5,1,1,5,43880.34,False


In [8]:
import sys
sys.path.append('../src')
from obtener_tipo_cambio import obtener_tipo_cambio

# Ejemplo de una sola consulta
obtener_tipo_cambio('2025-06-01')

{'amount': 1.0, 'base': 'EUR', 'date': '2025-05-30', 'rates': {'USD': 1.1339}}

In [9]:
df_pres = pd.read_sql('SELECT * FROM presupuesto', conn)

kpi = df_pres.merge(
    ejecucion_real_limpia[['departamento_id','categoria_id','periodo_id','monto_ejecutado']],
    on=['departamento_id','categoria_id','periodo_id'], how='left'
)
kpi = kpi.merge(df_periodos, on='periodo_id')
kpi = kpi.merge(df_departamentos, on='departamento_id').rename(columns={'nombre': 'departamento'})
kpi = kpi.merge(df_categorias, on='categoria_id').rename(columns={'nombre': 'categoria'})

kpi['sin_dato_ejecucion'] = kpi['monto_ejecutado'].isna()
kpi['pct_ejecucion'] = kpi['monto_ejecutado'] / kpi['monto_presupuestado']
kpi['desviacion_absoluta'] = kpi['monto_ejecutado'] - kpi['monto_presupuestado']
kpi['desviacion_pct'] = kpi['desviacion_absoluta'] / kpi['monto_presupuestado']

UMBRAL_ALERTA = 0.10
kpi['flag_alerta_sobreejecucion'] = kpi['desviacion_pct'] > UMBRAL_ALERTA

# Acumulado YTD por departamento y categoria
kpi = kpi.sort_values(['departamento_id', 'categoria_id', 'anio', 'mes'])
kpi['presupuesto_ytd'] = kpi.groupby(['departamento_id', 'categoria_id'])['monto_presupuestado'].cumsum()
kpi['ejecutado_ytd'] = kpi.groupby(['departamento_id', 'categoria_id'])['monto_ejecutado'].cumsum()
kpi['pct_ejecucion_ytd'] = kpi['ejecutado_ytd'] / kpi['presupuesto_ytd']

print(f"Filas KPI: {len(kpi)} | Sin dato de ejecucion: {kpi['sin_dato_ejecucion'].sum()}")
print(f"Alertas de sobreejecucion (>{UMBRAL_ALERTA:.0%}): {kpi['flag_alerta_sobreejecucion'].sum()}")

kpi[kpi['flag_alerta_sobreejecucion']][['departamento','categoria','mes','pct_ejecucion','desviacion_pct']].round(3).head(10)

Filas KPI: 240 | Sin dato de ejecucion: 10
Alertas de sobreejecucion (>10%): 15


,departamento,categoria,mes,pct_ejecucion,desviacion_pct
42,Marketing,Nomina y Personal,7,1.104,0.104
44,Marketing,Nomina y Personal,9,1.126,0.126
183,Marketing,Gastos Operativos,4,1.128,0.128
188,Marketing,Gastos Operativos,9,1.118,0.118
177,Marketing,Marketing y Publicidad,10,1.117,0.117
50,Tecnologia y TI,Nomina y Personal,3,1.100,0.100
51,Tecnologia y TI,Nomina y Personal,4,1.145,0.145
52,Tecnologia y TI,Nomina y Personal,5,1.112,0.112
55,Tecnologia y TI,Nomina y Personal,8,1.121,0.121
58,Tecnologia y TI,Nomina y Personal,11,1.130,0.130


In [10]:
df_ingresos = pd.read_sql('SELECT * FROM ingresos', conn)
df_ingresos.head()

,id_ingreso,departamento_id,concepto,periodo_id,monto_ingresos
0,1,2,Servicios B2B Corporativos,1,258950.55
1,2,2,Servicios B2B Corporativos,2,257646.06
2,3,2,Servicios B2B Corporativos,3,269513.74
3,4,2,Servicios B2B Corporativos,4,269992.46
4,5,2,Servicios B2B Corporativos,5,276210.22


In [11]:
ingresos_mes = (df_ingresos.merge(df_periodos, on='periodo_id')
                .groupby(['periodo_id', 'anio', 'mes'])['monto_ingresos'].sum()
                .reset_index(name='total_ingresos'))

gastos_mes = (ejecucion_real_limpia.merge(df_periodos, on='periodo_id')
              .groupby(['periodo_id', 'anio', 'mes'])['monto_ejecutado'].sum()
              .reset_index(name='total_gastos'))

resumen_mensual = ingresos_mes.merge(gastos_mes, on=['periodo_id', 'anio', 'mes']).sort_values('mes')
resumen_mensual['margen_operativo'] = resumen_mensual['total_ingresos'] - resumen_mensual['total_gastos']
resumen_mensual['margen_operativo_pct'] = resumen_mensual['margen_operativo'] / resumen_mensual['total_ingresos']
resumen_mensual['crecimiento_ingresos_mom'] = resumen_mensual['total_ingresos'].pct_change()

crecimiento_anual = resumen_mensual['total_ingresos'].iloc[-1] / resumen_mensual['total_ingresos'].iloc[0] - 1
print(f"Crecimiento de ingresos enero -> diciembre: {crecimiento_anual*100:.1f}%")
print(f"Margen operativo promedio: {resumen_mensual['margen_operativo_pct'].mean()*100:.1f}%")

resumen_mensual[['mes', 'total_ingresos', 'total_gastos', 'margen_operativo_pct']].round(3)

Crecimiento de ingresos enero -> diciembre: 18.3%
Margen operativo promedio: 29.1%


,mes,total_ingresos,total_gastos,margen_operativo_pct
0,1,475285.00,364072.30,0.234
1,2,479306.77,332116.98,0.307
2,3,494354.03,368224.44,0.255
3,4,495937.39,353320.50,0.288
4,5,510113.12,373165.18,0.268
5,6,505783.25,363579.91,0.281
6,7,495698.39,360503.00,0.273
7,8,466280.51,374899.55,0.196
8,9,534299.61,314041.18,0.412
9,10,535287.52,366279.79,0.316


In [12]:
# Crecimiento por linea de negocio (enero vs diciembre)
por_linea = df_ingresos.merge(df_periodos, on='periodo_id')
tabla_lineas = por_linea.pivot_table(index='mes', columns='concepto', values='monto_ingresos', aggfunc='sum')
crecimiento_lineas = ((tabla_lineas.loc[12] / tabla_lineas.loc[1] - 1) * 100).round(1)
crecimiento_lineas.name = 'crecimiento_pct_ene_dic'
crecimiento_lineas.sort_values(ascending=False)

concepto
Servicios de Valor Añadido (Cloud/Digital)    37.8
Servicios B2B Corporativos                    18.3
Servicios Residenciales                        9.4
Name: crecimiento_pct_ene_dic, dtype: float64